# 03 — Export Formats

Every `run_pipeline()` call already writes **CSV and JSON** to disk — that's the default.  
This notebook shows what those files look like and how to generate **Cypher** for Neo4j.

---
## Part A — CSV (already generated)

Every extraction run produces `nodes.csv` and `edges.csv` in Neo4j-compatible format.  
Let's look at the ones from the home insurance run in `01_quickstart`.

In [1]:
from pathlib import Path
import pandas as pd

# Find the most recent home policy export
runs = sorted(
    Path("../data/home_policy_output").glob("*/*/docling_graph"),
    key=lambda p: p.stat().st_mtime, reverse=True
)
if not runs:
    runs = sorted(
        Path("../data/prerun").glob("*"),
        key=lambda p: p.stat().st_mtime, reverse=True
    )

export_dir = runs[0]
print(f"Export dir: {export_dir}")
print("Files:", [f.name for f in export_dir.iterdir()])

Export dir: ../data/home_policy_output/qwen3.5-2b/sample-declarations-page_pdf_20260417_211758/docling_graph
Files: ['nodes.csv', 'report.md', 'graph.html', 'graph_summary.png', 'graph.json', 'edges.csv', 'graph_pretty.html']


In [2]:
nodes_csv = export_dir / "nodes.csv"
nodes_df = pd.read_csv(nodes_csv)

# Show only the populated columns
non_empty = [c for c in nodes_df.columns if nodes_df[c].notna().any() and nodes_df[c].astype(str).str.strip().ne('').any()]
print(f"nodes.csv — {len(nodes_df)} rows")
nodes_df[non_empty].fillna('').head(10)

nodes.csv — 14 rows


,id,label,type,__class__,name,address,phone,agent_number,coverage_code,coverage_name,limit,premium,peril_type,amount,policy_number,policy_form,effective_date,expiration_date,total_premium,property_address
0,Insurer_3cf7f5aee6d9d864,Insurer,entity,Insurer,Safe All Insurance Company,"P.O. Box 1075 Thiston, FL 12345-6789",(123) 456-7891,,,,,,,,,,,,,
1,Insured_ed314a4620ad85a1,Insured,entity,Insured,Estelle Clarion & James Delaney,"596 Crossover Way Clay, FL 17189-2021",,,,,,,,,,,,,,
2,Agent_8df62d5d5774f33c,Agent,entity,Agent,Tony Prize,,(123) 456-7891,194722.0,,,,,,,,,,,,
3,Coverage_ed0313f20cf00963,Coverage,entity,Coverage,,,,,Coverage A,Dwelling,"$160,000",$859.00,,,,,,,,
4,Coverage_221d31d3b213e83a,Coverage,entity,Coverage,,,,,Coverage B,Other Structures,"$3,200",Included,,,,,,,,
5,Coverage_0d26db320e8c04ff,Coverage,entity,Coverage,,,,,Coverage C,Personal Property,"$104,250",Included,,,,,,,,
6,Coverage_5dd5783c4ef23b6a,Coverage,entity,Coverage,,,,,Coverage D,Loss of Use,"$20,850",Included,,,,,,,,
7,Coverage_86d59db006f0781a,Coverage,entity,Coverage,,,,,Coverage E,Personal Liability,"$100,000",Included,,,,,,,,
8,Coverage_9c4a2895b3d284f8,Coverage,entity,Coverage,,,,,Coverage F,Medical Payments,"$1,000",Included,,,,,,,,
9,Coverage_18cadb998f54d2fd,Coverage,entity,Coverage,,,,,Water Backup and Sump Overflow,Water Backup and Sump Overflow,"$5,000",$25.00,,,,,,,,


In [3]:
edges_csv = export_dir / "edges.csv"
edges_df = pd.read_csv(edges_csv)
print(f"edges.csv — {len(edges_df)} rows")
edges_df

edges.csv — 13 rows


,source,target,label
0,HomePolicy_c88f1404047cbd8c,Insurer_3cf7f5aee6d9d864,insurer
1,HomePolicy_c88f1404047cbd8c,Insured_ed314a4620ad85a1,insured
2,HomePolicy_c88f1404047cbd8c,Agent_8df62d5d5774f33c,agent
3,HomePolicy_c88f1404047cbd8c,Coverage_ed0313f20cf00963,coverages
4,HomePolicy_c88f1404047cbd8c,Coverage_221d31d3b213e83a,coverages
5,HomePolicy_c88f1404047cbd8c,Coverage_0d26db320e8c04ff,coverages
6,HomePolicy_c88f1404047cbd8c,Coverage_5dd5783c4ef23b6a,coverages
7,HomePolicy_c88f1404047cbd8c,Coverage_86d59db006f0781a,coverages
8,HomePolicy_c88f1404047cbd8c,Coverage_9c4a2895b3d284f8,coverages
9,HomePolicy_c88f1404047cbd8c,Coverage_18cadb998f54d2fd,coverages


These two files are the Neo4j **bulk import** format — drop them into `neo4j-admin database import full`  
or load them into any tool that reads CSV (Pandas, DuckDB, Spark).

---
## Part B — Cypher Export

`docling-graph` ships a `CypherExporter` that generates `CREATE` statements for Neo4j.  
We call it directly on the loaded graph — no re-extraction needed.

In [4]:
import json
import networkx as nx

def load_graph(path: Path) -> nx.DiGraph:
    data = json.loads(path.read_text())
    G = nx.DiGraph()
    for n in data["nodes"]:
        n = dict(n); nid = n.pop("id"); G.add_node(nid, **n)
    for e in data["edges"]:
        G.add_edge(e["source"], e["target"],
                   relation=e.get("relation") or e.get("label", ""))
    return G

json_path = export_dir / "graph.json"
G = load_graph(json_path)
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

14 nodes, 13 edges


In [5]:
from docling_graph.core.exporters.cypher_exporter import CypherExporter

cypher_path = export_dir / "graph.cypher"
CypherExporter().export(G, cypher_path)
print(f"Saved: {cypher_path}")

Saved: ../data/home_policy_output/qwen3.5-2b/sample-declarations-page_pdf_20260417_211758/docling_graph/graph.cypher


In [6]:
cypher = cypher_path.read_text()
print(cypher)

// Cypher script generated by docling-graph
// Import this into Neo4j

// --- Create Nodes ---
CREATE (Insurer_3cf7f5aee6d9d864_0:Insurer {type: "entity", __class__: "Insurer", name: "Safe All Insurance Company", address: "P.O. Box 1075 Thiston, FL 12345-6789", phone: "(123) 456-7891"})
CREATE (Insured_ed314a4620ad85a1_1:Insured {type: "entity", __class__: "Insured", name: "Estelle Clarion & James Delaney", address: "596 Crossover Way Clay, FL 17189-2021"})
CREATE (Agent_8df62d5d5774f33c_2:Agent {type: "entity", __class__: "Agent", name: "Tony Prize", agent_number: "194722", phone: "(123) 456-7891"})
CREATE (Coverage_ed0313f20cf00963_3:Coverage {type: "entity", __class__: "Coverage", coverage_code: "Coverage A", coverage_name: "Dwelling", limit: "$160,000", premium: "$859.00"})
CREATE (Coverage_221d31d3b213e83a_4:Coverage {type: "entity", __class__: "Coverage", coverage_code: "Coverage B", coverage_name: "Other Structures", limit: "$3,200", premium: "Included"})
CREATE (Coverage_0d26db

---
## Format reference

| Format | Files | Best for | How to import |
|---|---|---|---|
| **CSV** (default) | `nodes.csv`, `edges.csv` | Production bulk import | `neo4j-admin database import full` |
| **Cypher** | `graph.cypher` | Dev, sandbox, incremental | Paste into Neo4j Browser |
| **JSON** | `graph.json` | API integration, Python | `json.load()` → NetworkX |

Switch format in `PipelineConfig`:
```python
config = PipelineConfig(
    ...
    export_format="cypher",   # or "csv" (default)
)
```

All three formats are also **always** generated when `dump_to_disk=True`.